# Scaling Ambiguity Augmenting Human Annotation in Speech Emotion Recognition with Audio-Language Models - Synthetic Annotation Generation

This notebook generates synthetic annotation using **Gemini-2.5 Pro** for **IEMOCAP** and **MSP-Podcast datasets**. Following the methodology, these annotations will serve as the **Synthetic Perceptual Proxies**.

## Dependencies Import

In [ ]:
# Dependency Imports
import sys
sys.path.append('..') 

import json
import os
import time
import numpy as np
import random
import datetime
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from google import genai
from google.genai import types
from collections import Counter

from lib import load_data as ld

# Section 1: Load Training Data

In [ ]:
# Load the training datasets that were processed in the `data_processing.ipynb` notebook
iemocap_train = ld.load_train_distributions('iemocap')
msp_train = ld.load_train_distributions('msp')

iemocap_test = ld.load_test_distributions('iemocap')
msp_test = ld.load_test_distributions('msp')

iemocap_emotions = ld.load_emotion_classes('iemocap')
msp_emotions = ld.load_emotion_classes('msp')
  
# Print emotion classes
print("\nIEMOCAP emotion classes:")
print(iemocap_emotions)
    
print("\nMSP-Podcast emotion classes:")
print(msp_emotions)

Loaded 3496 human training samples from IEMOCAP dataset
Loaded 3291 human training samples from MSP dataset
Loaded 874 test samples from IEMOCAP dataset
Loaded 823 test samples from MSP dataset
Loaded 4 emotion classes for IEMOCAP dataset
Loaded 4 emotion classes for MSP dataset

IEMOCAP emotion classes:
['Anger', 'Happiness', 'Neutral state', 'Sadness']

MSP-Podcast emotion classes:
['Angry', 'Happy', 'Neutral', 'Sad']


# Section 2: API Setup

In [ ]:
# Initialize GenAI client
load_dotenv()
client = genai.Client(
    api_key=os.environ.get("GENAI_API_KEY")
)

def call_gemini_with_audio(prompt, audio_path, temperature=None, max_retries=3, retry_delay=2):
    """
    Call Gemini API with both text prompt and audio file
    
    Parameters:
    - prompt: The text prompt to send to Gemini
    - audio_path: Path to the audio file
    - temperature: Temperature setting for Gemini (higher = more variability)
    - max_retries: Maximum number of retry attempts
    - retry_delay: Delay between retries in seconds
    
    Returns:
    - The response from Gemini or None if all retries fail
    """
    
    if temperature is None:
        temperature = 0.3  # Default temperature for emotion analysis
        
    for attempt in range(max_retries):
        try:
            # Read the audio file as bytes
            with open(audio_path, "rb") as audio_file:
                audio_bytes = audio_file.read()
            
            # Create parts with text and file data
            text_part = {"text": prompt}
            
            # Get file MIME type
            mime_type = "audio/wav"
            
            # Add file as byte data
            file_part = {
                "inline_data": {
                    "mime_type": mime_type,
                    "data": audio_bytes
                }
            }
            
            # Generate content with the correct format
            response = client.models.generate_content(
                model="gemini-2.5-pro-preview-05-06",
                contents=[
                    text_part,
                    file_part
                ],
                config=genai.types.GenerateContentConfig(
                    system_instruction="You are an expert in emotion analysis who provides accurate emotion judgments.",
                    temperature=temperature
                ),
            )
            
            # Extract text from the response
            return response.text
            
        except FileNotFoundError as e:
            print(f"File error: {str(e)}")
            return None
        except Exception as e:
            print(f"Error on attempt {attempt+1}/{max_retries}: {str(e)}")
            if attempt < max_retries - 1:
                print(f"Retrying in {retry_delay} seconds...")
                time.sleep(retry_delay)
                retry_delay *= 1.5
            else:
                print("All retry attempts failed.")
                return None

# Section 3: Design Annotation Prompt

In [14]:
def create_iemocap_single_emotion_prompt(sample, annotator_id):
    """
    Create a prompt for Gemini-2.5 Pro to choose a single emotion for IEMOCAP samples
    
    Parameters:
    - sample: The sample to annotate
    - annotator_id: Simulated annotator ID to create slight variations in prompts
    """
    # List all possible emotions for IEMOCAP
    emotion_list = ", ".join(iemocap_emotions)
    
    # Add slight variations based on annotator_id
    focus_variations = [
        "Focus on both vocal tone and linguistic content.",
        "Pay special attention to the speaker's tone of voice.",
        "Consider the word choice and phrasing in the utterance.",
        "Listen for subtle emotional cues in the voice.",
        "Analyze both what is said and how it is expressed."
    ]
    
    # Select a focus variation based on annotator_id
    focus = focus_variations[annotator_id % len(focus_variations)]
    
    prompt = f"""Listen to the audio and analyze this speech utterance. Think deeply and Select the ONE most dominant emotion present.

Transcript: "{sample['groundtruth']}"
Speaker: {sample['speaker']}

Possible emotions: {emotion_list}

Instructions:
1. Carefully analyze the emotional content in both the audio and the text.
2. {focus}
3. Think deeply and Select ONLY ONE emotion from the list that best represents the dominant emotional state.
4. Respond with ONLY the emotion name, nothing else.

Which single emotion best describes this utterance? Select only from: {emotion_list}"""
    
    # Get audio path
    audio_path = ld.get_audio_path(sample, 'iemocap')
    
    return prompt, audio_path

def create_msp_single_emotion_prompt(sample, annotator_id):
    """
    Create a prompt for Gemini-2.5 Pro to choose a single emotion for MSP samples
    
    Parameters:
    - sample: The sample to annotate
    - annotator_id: Simulated annotator ID to create slight variations in prompts
    """
    # List all possible emotions for MSP
    emotion_list = ", ".join(msp_emotions)
    
    # Handle groundtruth which might be a list or string
    if isinstance(sample['groundtruth'], list):
        if sample['groundtruth']:  # Check if list is not empty
            # Extract the string from the list and remove any extra quotes
            raw_text = sample['groundtruth'][0]
            # Check if the extracted string still has list formatting
            if isinstance(raw_text, str) and raw_text.startswith('[') and raw_text.endswith(']'):
                # Try to interpret as a string representation of a list
                try:
                    # Using ast.literal_eval is safer than eval()
                    import ast
                    parsed = ast.literal_eval(raw_text)
                    if isinstance(parsed, list) and parsed:
                        utterance = parsed[0]
                    else:
                        utterance = raw_text
                except:
                    # If parsing fails, use as is
                    utterance = raw_text
            else:
                utterance = raw_text
        else:
            # Empty list - no transcript available
            utterance = None
    else:
        utterance = sample['groundtruth']
    
    # Add slight variations based on annotator_id
    focus_variations = [
        "Focus on both vocal tone and linguistic content.",
        "Pay special attention to the speaker's tone of voice.",
        "Consider the word choice and phrasing in the utterance.",
        "Listen for subtle emotional cues in the voice.",
        "Analyze both what is said and how it is expressed."
    ]
    
    # Select a focus variation based on annotator_id
    focus = focus_variations[annotator_id % len(focus_variations)]
    
    # Base prompt template
    base_prompt = """Listen to the audio and analyze this speech utterance. Think deeply and Select the ONE most dominant emotion present.

Utterance: {transcript_section}
Speaker: {speaker}

Possible emotions: {emotion_list}

Instructions:
1. {primary_instruction}
2. {focus}
3. Think deeply and Select ONLY ONE emotion from the list that best represents the dominant emotional state.
4. Respond with ONLY the emotion name, nothing else.

Which single emotion best describes this utterance? Select only from: {emotion_list}"""
    
    # Determine transcript section and primary instruction based on availability
    if utterance is None:
        transcript_section = ""  # Skip transcript line entirely
        primary_instruction = "This audio has no utterance. Rely solely on audio cues for analysis."
    else:
        # Clean any lingering quotes or whitespace
        clean_utterance = utterance.strip()
        transcript_section = f'"{clean_utterance}"'
        primary_instruction = "Carefully analyze the emotional content in both the audio and the text."
    
    # Format the complete prompt
    prompt = base_prompt.format(
        transcript_section=transcript_section,
        speaker=sample['speaker'],
        emotion_list=emotion_list,
        primary_instruction=primary_instruction,
        focus=focus
    )
    
    # Get audio path
    audio_path = ld.get_audio_path(sample, 'msp')
    
    return prompt, audio_path

In [ ]:
# Test the prompt generation
print("\nSample IEMOCAP single emotion prompt:")
test_prompt, _ = create_iemocap_single_emotion_prompt(iemocap_train[0], 0)
print(test_prompt)

In [ ]:
# Test the prompt generation
print("\nSample MSP single emotion prompt:")
test_prompt, _ = create_msp_single_emotion_prompt(msp_train[127], 1)
print(test_prompt)

# Section 4: Generate Annotations

In [17]:
def parse_single_emotion(response_text, valid_emotions):
    """
    Parse the Gemini-2.5 Pro response to extract a single emotion.
    
    Parameters:
    - response_text: Response from Gemini
    - valid_emotions: List of valid emotion categories for this dataset
    
    Returns:
    - The extracted emotion or None if parsing failed
    """
    if not response_text:
        return None
    
    # Clean and normalize the response
    cleaned_text = response_text.strip().lower()
    
    # Try exact matching first
    for emotion in valid_emotions:
        if cleaned_text == emotion.lower():
            return emotion
    
    # Try to find any of the valid emotions in the response
    for emotion in valid_emotions:
        if emotion.lower() in cleaned_text:
            return emotion
    
    print(f"Could not parse emotion from response: '{response_text}'")
    print(f"Valid emotions: {valid_emotions}")
    return None

In [18]:
def get_diverse_temperature(annotator_id, num_annotators):
    """
    Generates a more evenly distributed temperature value, ensuring a balanced ratio of low/medium/high temperatures
    """
    # Use annotator_id as a random seed to ensure that the same annotator always gets the same temperature
    np.random.seed(annotator_id)

    # Calculate which temperature interval this annotator should be in
    # Divide the temperature range 0.1-1.0 evenly among all annotators
    segment = annotator_id % 5 # Divide into 5 temperature intervals

    if segment == 0:
        # Low temperature interval (0.1-0.3)
        base_temp = 0.1 + np.random.uniform(0, 0.2)
    elif segment == 1:
        # Medium and low temperature interval (0.3-0.5)
        base_temp = 0.3 + np.random.uniform(0, 0.2)
    elif segment == 2:
        # Medium temperature interval (0.5-0.7)
        base_temp = 0.5 + np.random.uniform(0, 0.2)
    elif segment == 3:
        # Medium and high temperature range (0.7-0.9)
        base_temp = 0.7 + np.random.uniform(0, 0.2)
    else: # segment == 4
        # High temperature range (0.9-0.93)
        base_temp = 0.9 + np.random.uniform(0, 0.03)

    # Add a small amount of random fluctuation (±0.05), but keep it within the range
    jitter = np.random.uniform(-0.05, 0.05)
    temp = max(0.1, min(1.0, base_temp + jitter))

    return temp

In [ ]:
def generate_multiple_annotations(dataset, dataset_name, create_prompt_func, num_annotators=5, batch_size=5, max_samples=None):
    """
    Generate multiple Gemini-2.5 Pro annotations per sample to simulate multiple human annotators
    
    Parameters:
    - dataset: The dataset to annotate
    - dataset_name: Name of the dataset (for logging)
    - create_prompt_func: Function to create prompts for this dataset
    - num_annotators: Number of simulated annotators per sample
    - batch_size: Number of samples to process before saving progress
    - max_samples: Maximum number of samples to process (for testing/development)
    
    Returns:
    - Annotated dataset with both raw annotations and aggregated distributions
    """
    # Create output directories
    root_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
    output_dir = os.path.join(root_dir, 'processed_data', 'gemini_annotations')
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Prepare output files
    raw_annotations_file = os.path.join(output_dir, f'gemini_{dataset_name}_test_raw_annotations.json')
    distribution_file = os.path.join(output_dir, f'gemini_{dataset_name}_test_distributions.json')
    progress_file = os.path.join(output_dir, f'{dataset_name}_progress.json')
    
    # Load progress if exists
    raw_annotations = {}
    if os.path.exists(raw_annotations_file):
        with open(raw_annotations_file, 'r') as f:
            raw_annotations = json.load(f)
        print(f"Loaded existing annotations for {len(raw_annotations)} {dataset_name} samples")
    
    # Determine samples to process
    processed_ids = set(raw_annotations.keys())
    remaining_samples = [s for s in dataset if s['id'] not in processed_ids]
    
    # Limit samples if needed
    if max_samples is not None:
        remaining_samples = remaining_samples[:max_samples]
    
    # Determine valid emotions for dataset
    valid_emotions = iemocap_emotions if dataset_name == 'iemocap' else msp_emotions
    
    print(f"Generating annotations for {len(remaining_samples)} remaining {dataset_name} samples")
    
    # Process samples
    for sample_idx, sample in enumerate(tqdm(remaining_samples, desc=f"Processing {dataset_name} samples")):
        sample_id = sample['id']
        raw_annotations[sample_id] = []
        
        # For each sample, simulate multiple annotators
        for annotator_id in range(num_annotators):
            
            # Get a diverse temperature for this annotator
            temperature = get_diverse_temperature(annotator_id, num_annotators)
            
            # Create prompt with slight variations
            prompt, audio_path = create_prompt_func(sample, annotator_id)
            
            # Check if audio file exists
            if not os.path.exists(audio_path):
                print(f"Audio file not found: {audio_path}")
                continue
            
           # Add delay to avoid rate limiting
            if annotator_id > 0:
                time.sleep(random.uniform(3, 6)) # Randomly wait time to avoid synchronous requests
                
            response_text = call_gemini_with_audio(prompt, audio_path, temperature=temperature)
            
            if response_text:
                # Parse the emotion from response
                emotion = parse_single_emotion(response_text, valid_emotions)
                
                if emotion:
                    # Record this annotation
                    raw_annotations[sample_id].append({
                        "annotator_id": f"gemini_{annotator_id}",
                        "emotion": emotion,
                        "response": response_text,
                        "temperature": temperature
                    })
        
        # Save progress periodically
        if (sample_idx + 1) % batch_size == 0 or sample_idx == len(remaining_samples) - 1:
            with open(raw_annotations_file, 'w') as f:
                json.dump(raw_annotations, f, indent=2)
                
            progress_info = {
                "total_samples": len(dataset),
                "processed_samples": len(raw_annotations),
                "remaining_samples": len(dataset) - len(raw_annotations),
                "last_processed_id": sample_id,
                "timestamp": datetime.datetime.now().isoformat()
            }
                
            with open(progress_file, 'w') as f:
                json.dump(progress_info, f, indent=2)
                    
            print(f"Progress saved: {len(raw_annotations)}/{len(dataset)} {dataset_name} samples")
    
    # Convert raw annotations to distribution format
    distribution_data = []
    
    for sample in dataset:
        sample_id = sample['id']
        
        # Create a copy of the sample for distribution data
        sample_copy = dict(sample)
        
        # Check if we have annotations for this sample
        if sample_id in raw_annotations and raw_annotations[sample_id]:
            # Extract emotions
            emotions = [a["emotion"] for a in raw_annotations[sample_id] if "emotion" in a]
            
            if emotions:
                # Count occurrences of each emotion
                emotion_counts = Counter(emotions)
                # Convert to distribution
                total = len(emotions)
                emotion_distribution = {emotion: count/total for emotion, count in emotion_counts.items()}
                
                # Replace emotion field with distribution
                sample_copy['emotion'] = emotion_distribution
                sample_copy['annotation_source'] = 'gemini-2.5-pro-preview-05-06'
                sample_copy['num_annotators'] = len(emotions)
                
                distribution_data.append(sample_copy)
    
    # Save distribution data
    with open(distribution_file, 'w') as f:
        json.dump(distribution_data, f, indent=2)
    
    print(f"Completed Gemini-2.5 Pro annotation for {dataset_name}: {len(distribution_data)} samples with distributions")
    
    return distribution_data, raw_annotations

# Section 5: Run the Annotation Process

In [ ]:
max_samples = 1 # Set to 'None' for full dataset

# Generate annotations for IEMOCAP
iemocap_distributions, iemocap_raw = generate_multiple_annotations(
    dataset=iemocap_test,
    dataset_name='iemocap',
    create_prompt_func=create_iemocap_single_emotion_prompt,
    # Simulate 8 annotators per sample
    num_annotators=8,
    batch_size=3,
    max_samples=max_samples
)

# Generate annotations for MSP-Podcast
msp_distributions, msp_raw = generate_multiple_annotations(
    dataset=msp_train,
    dataset_name='msp',
    create_prompt_func=create_msp_single_emotion_prompt,
# Simulate 10 annotators per sample
    num_annotators=10,
    batch_size=3,
    max_samples=max_samples
)